In [1]:
!pip3 install BeautifulSoup4
!pip3 install --upgrade pip

In [2]:
import os
import json
import time
import random
import zipfile
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Class Explanation: `NewsScraper`

## Overview
The `NewsScraper` class is designed for scraping news articles from three different Urdu news websites: Geo, Jang, and Express. The class has methods that cater to each site's unique structure and requirements. Below, we will go through the class and its methods, detailing what each function does, the input it takes, and the output it returns.

## Class Definition

```python
class NewsScraper:
    def __init__(self, id_=0):
        self.id = id_
```


## Method 1: `get_express_articles`

### Description
Scrapes news articles from the Express website across categories like saqafat (entertainment), business, sports, science-technology, and world. The method navigates through multiple pages for each category to gather a more extensive dataset.

### Input
- **`max_pages`**: The number of pages to scrape for each category (default is 7).

### Process
- Iterates over each category and page.
- Requests each category page and finds article cards within `<ul class='tedit-shortnews listing-page'>`.
- Extracts the article's headline, link, and content by navigating through `<div class='horiz-news3-caption'>` and `<span class='story-text'>`.

### Output
- **Returns**: A tuple of:
  - A Pandas DataFrame containing columns: `id`, `title`, and `link`).
  - A dictionary `express_contents` where the key is the article ID and the value is the article content.

### Data Structure
- Article cards are identified by `<li>` tags.
- Content is structured within `<span class='story-text'>` and `<p>` tags.



In [3]:
class NewsScraper:
    def __init__(self,id_=0):
        self.id = id_


  # write functions to scrape from other websites


    def get_express_articles(self, max_pages=8):
        express_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://www.express.pk'
        categories = ['saqafat', 'business', 'sports', 'science', 'world']   # saqafat is entertainment category

        # Iterating over the specified number of pages
        for category in categories:
            for page in range(1, max_pages + 1):
                print(f"Scraping page {page} of category '{category}'...")
                url = f"{base_url}/{category}/archives?page={page}"
                response = requests.get(url)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                # Finding article cards
                cards = soup.find('ul', class_='tedit-shortnews listing-page').find_all('li')  # Adjust class as per actual site structure
                print(f"\t--> Found {len(cards)} articles on page {page} of '{category}'.")

                success_count = 0

                for card in cards:
                    try:
                        div = card.find('div',class_='horiz-news3-caption')

                        # Article Title
                        headline = div.find('a').get_text(strip=True).replace('\xa0', ' ')

                        # Article link
                        link = div.find('a')['href']

                        # Requesting the content from each article's link
                        article_response = requests.get(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(article_response.text, "html.parser")


                        # Content arranged in paras inside <span> tags
                        paras = content_soup.find('span',class_='story-text').find_all('p')

                        combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                        )

                        # Storing data
                        express_df['id'].append(self.id)
                        express_df['title'].append(headline)
                        express_df['link'].append(link)
                        express_df['gold_label'].append(category.replace('saqafat','entertainment').replace('science','science-technology'))
                        express_df['content'].append(combined_text)

                        # Increment ID and success count
                        self.id += 1
                        success_count += 1

                    except Exception as e:
                        print(f"\t--> Failed to scrape an article on page {page} of '{category}': {e}")

                print(f"\t--> Successfully scraped {success_count} articles from page {page} of '{category}'.")
            print('')

        return pd.DataFrame(express_df)
    
    def get_ge0_articles(self, max_pages=1):
        geo_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        
        base_url = 'https://urdu.geo.tv'
        categories = ['entertainment', 'business', 'sports', 'science-technology', 'world']   # showbiz is entertainment category

        # Iterating over the specified number of pages
        for category in categories:
            # for page in range(1):
                print(f"Scraping category '{category}'...")
                url = f"{base_url}/category/{category}"
                print(url)
                response = requests.get(url)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                # Finding article cards
                divs = soup.find_all('div', class_='col-xs-6 col-sm-6 col-lg-6 col-md-6 singleBlock') # Adjust class as per actual site structure
                print(f"\t--> Found {len(divs)} articles of '{category}'.")

                success_count = 0

                for div in divs:
                    try:
                        # div = card.find('div',class_='horiz-news3-caption')

                        # Article Title
                        title = div.find('h2', {'data-vr-headline': True}).get_text(strip=True).replace('\xa0',' ')
                        
                        # headline = div.find('a').get_text(strip=True).replace('\xa0', ' ')

                        # Article link
                        
                        link = div.find('a')['href']
                        article_response = requests.get(link)
                        # print(title)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(article_response.text, "html.parser")
                        

                        # Content arranged in paras inside <div> tags
                        paras = content_soup.find('div',class_='content-area').find_all('p')

                        combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                        )
                        # Storing data
                        geo_df['id'].append(self.id)
                        geo_df['title'].append(title)
                        geo_df['link'].append(link)
                        geo_df['gold_label'].append(category)
                        geo_df['content'].append(combined_text)

                        # Increment ID and success count
                        self.id += 1
                        success_count += 1
                    except Exception as e:
                        print(f"\t--> Failed to scrape an article of '{category}': {e}")

                print(f"\t--> Successfully scraped {success_count} articles from page  of '{category}'.")
        return pd.DataFrame(geo_df)
    
    def get_jang_articles(self, max_pages=1):
        geo_df = {
            "id": [],
            "title": [],
            "link": [],
            "content": [],
            "gold_label": [],
        }
        base_url = 'https://jang.com.pk/category/latest-news'
        categories = ['entertainment', 'business', 'sports', 'science-and-technology', 'world']   #y

        # Iterating over the specified number of pages
        for category in categories:
            # for page in range(1):
                print(f"Scraping category '{category}'...")
                url = f"{base_url}/{category}"
                print(url)
                response = requests.get(url)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, "html.parser")

                # Finding article cards
                main = soup.find('div', class_='latest_page_right') # Adjust class as per actual site structure
                divs=main.find_all('li',class_=lambda x: x != 'ad_latest_stories')
                print(f"\t--> Found {len(divs)} articles of '{category}'.")

                success_count = 0

                for div in divs:
                    try:
                        # div = card.find('div',class_='horiz-news3-caption')

                        # Article Title
                        # title = div.find('a').get_text(strip=True).replace('\xa0', ' ')
                        title = div.find('h2').get_text(strip=True).replace('\xa0',' ')
                        # print(title)
                        # headline = div.find('a').get_text(strip=True).replace('\xa0', ' ')

                        # Article link
                        
                        link = div.find('a')['href']
                        article_response = requests.get(link)
                        
                        # print(link)
                        article_response.raise_for_status()
                        content_soup = BeautifulSoup(article_response.text, "html.parser")
                        

                        # Content arranged in paras inside <div> tags
                        paras = content_soup.find('div',class_='detail_view_content').find_all('p')

                        combined_text = " ".join(
                        p.get_text(strip=True).replace('\xa0', ' ').replace('\u200b', '')
                        for p in paras if p.get_text(strip=True)
                        )
                        # Storing data
                        geo_df['id'].append(self.id)
                        geo_df['title'].append(title)
                        geo_df['link'].append(link)
                        geo_df['gold_label'].append(category.replace('science-and-technology','science-technology'))
                        geo_df['content'].append(combined_text)

                        # Increment ID and success count
                        self.id += 1
                        success_count += 1
                    except Exception as e:
                        print(f"\t--> Failed to scrape an article of '{category}': {e}")

                print(f"\t--> Successfully scraped {success_count} articles from page  of '{category}'.")
        return pd.DataFrame(geo_df)
            
            
            

        
    
        


In [4]:
scraper = NewsScraper()

In [5]:

geodata=scraper.get_ge0_articles()

Scraping category 'entertainment'...
https://urdu.geo.tv/category/entertainment
	--> Found 60 articles of 'entertainment'.
	--> Successfully scraped 60 articles from page  of 'entertainment'.
Scraping category 'business'...
https://urdu.geo.tv/category/business
	--> Found 60 articles of 'business'.
	--> Successfully scraped 60 articles from page  of 'business'.
Scraping category 'sports'...
https://urdu.geo.tv/category/sports
	--> Found 60 articles of 'sports'.
	--> Successfully scraped 60 articles from page  of 'sports'.
Scraping category 'science-technology'...
https://urdu.geo.tv/category/science-technology
	--> Found 60 articles of 'science-technology'.
	--> Successfully scraped 60 articles from page  of 'science-technology'.
Scraping category 'world'...
https://urdu.geo.tv/category/world
	--> Found 60 articles of 'world'.
	--> Successfully scraped 60 articles from page  of 'world'.


In [7]:
express=scraper.get_express_articles()

Scraping page 1 of category 'saqafat'...
	--> Found 10 articles on page 1 of 'saqafat'.
	--> Successfully scraped 10 articles from page 1 of 'saqafat'.
Scraping page 2 of category 'saqafat'...
	--> Found 10 articles on page 2 of 'saqafat'.
	--> Successfully scraped 10 articles from page 2 of 'saqafat'.
Scraping page 3 of category 'saqafat'...
	--> Found 10 articles on page 3 of 'saqafat'.
	--> Successfully scraped 10 articles from page 3 of 'saqafat'.
Scraping page 4 of category 'saqafat'...
	--> Found 10 articles on page 4 of 'saqafat'.
	--> Successfully scraped 10 articles from page 4 of 'saqafat'.
Scraping page 5 of category 'saqafat'...
	--> Found 10 articles on page 5 of 'saqafat'.
	--> Successfully scraped 10 articles from page 5 of 'saqafat'.
Scraping page 6 of category 'saqafat'...
	--> Found 10 articles on page 6 of 'saqafat'.
	--> Successfully scraped 10 articles from page 6 of 'saqafat'.
Scraping page 7 of category 'saqafat'...
	--> Found 10 articles on page 7 of 'saqafat'.


In [8]:

jangdata=scraper.get_jang_articles()

Scraping category 'entertainment'...
https://jang.com.pk/category/latest-news/entertainment
	--> Found 100 articles of 'entertainment'.
	--> Successfully scraped 100 articles from page  of 'entertainment'.
Scraping category 'business'...
https://jang.com.pk/category/latest-news/business
	--> Found 98 articles of 'business'.
	--> Successfully scraped 98 articles from page  of 'business'.
Scraping category 'sports'...
https://jang.com.pk/category/latest-news/sports
	--> Found 99 articles of 'sports'.
	--> Successfully scraped 99 articles from page  of 'sports'.
Scraping category 'science-and-technology'...
https://jang.com.pk/category/latest-news/science-and-technology
	--> Found 100 articles of 'science-and-technology'.
	--> Successfully scraped 100 articles from page  of 'science-and-technology'.
Scraping category 'world'...
https://jang.com.pk/category/latest-news/world
	--> Found 98 articles of 'world'.
	--> Successfully scraped 98 articles from page  of 'world'.


# Output
- Save a combined csv of all 3 sites.

In [9]:
# num_words = len(geodata.loc[15, 'content'].split())
# print("Number of words:", num_words)


# jangdata['id']=range(700,1193)
# print(jangdata)

combined_df = pd.concat([geodata,express,jangdata])

# Save the combined DataFrame to a new CSV file
combined_df.to_csv('combined_file.csv', index=False)

